In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [2]:
!pip install kagglehub

In [3]:
# load dataset

df = pd.read_csv("Churn_Modelling.csv")

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df.drop(columns=['RowNumber','Surname','CustomerId'],axis=1,inplace=True)

In [6]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
# encoding categorical variable

label_encoder_gender =LabelEncoder()
df['Gender'] = label_encoder_gender.fit_transform(df['Gender'])

In [8]:
label_encoder_geographical  = LabelEncoder() 

In [9]:
## onehot encoding in geography

from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geography = OneHotEncoder()

geo_encoder=one_hot_encoder_geography.fit_transform(df[['Geography']])

In [10]:
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [11]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [12]:
one_hot_encoder_geography.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [13]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=one_hot_encoder_geography.get_feature_names_out(['Geography']))

In [14]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [15]:
# Columns one hot encoder columns with the original data
df = pd.concat([df.drop("Geography",axis=1),geo_encoded_df],axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [16]:
## save the encoder and sacaler

with open('label_encoder_gender.pkl','wb')as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geography,file)

In [17]:
# divide the dataset into independent and dependent features

X= df.drop('Exited',axis=1)
y=df['Exited']

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=15)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
with open('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)

### ANN implementation


In [20]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [21]:
## Build our ann model

model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## first hiden layer (H1) connected with input layer
    Dense(32,activation='relu'), ## second layer
    Dense(1,activation='sigmoid') ## output layer
])

d:\use\padhle\python\projects\deeplearning\churn_modeling\venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [24]:
## compile the model

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [25]:
## setup the tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir = 'logs/fit'+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(log_dir = log_dir,histogram_freq=1)

In [26]:
## setup early stopping
early_stoping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)


In [27]:
### stage of training the model
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stoping_callback]
)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8309 - loss: 0.3966 - val_accuracy: 0.8488 - val_loss: 0.3674
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8568 - loss: 0.3543 - val_accuracy: 0.8532 - val_loss: 0.3563
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step - accuracy: 0.8577 - loss: 0.3468 - val_accuracy: 0.8472 - val_loss: 0.3597
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8573 - loss: 0.3446 - val_accuracy: 0.8528 - val_loss: 0.3551
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8632 - loss: 0.3367 - val_accuracy: 0.8592 - val_loss: 0.3461
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8615 - loss: 0.3326 - val_accuracy: 0.8488 - val_loss: 0.3637
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8663 - loss: 0.3295 - val_accuracy: 0.8504 - val_loss: 0.3730
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8629 - loss: 0.3293 - val_ac

In [28]:
model.save('model.h5')

In [29]:
## load tensorboard Extension
%load_ext tensorboard

In [30]:
%tensorboard --logdir logs/fit

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "D:\use\padhle\python\projects\deeplearning\churn_modeling\venv\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "D:\use\padhle\python\projects\deeplearning\churn_modeling\venv\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "D:\use\padhle\python\projects\deeplearning\churn_modeling\venv\Scripts\tensorboard.exe\__main__.py", line 2, in <module>
  File "D:\use\padhle\python\projects\deeplearning\churn_modeling\venv\lib\site-packages\tensorboard\main.py", line 27, in <module>
    from tensorboard import default
  File "D:\use\padhle\python\projects\deeplearning\churn_modeling\venv\lib\site-packages\tensorboard\default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'